# Silver: parse the PDFs with `ai_parse_document`

Task `silver_parse` of the job `ingest_financebench`. Reads the parts in `bronze_pdf_parts` as a stream, runs the AI
parser on each one and saves the result in `silver_parsed`, **one row per part**:

| Column | Holds |
| --- | --- |
| `doc_name` | The document name, from the file name: `3M_2018_10K` for `.../pdfs/3M_2018_10K.pdf`. It is the `doc_name` FinanceBench's questions use |
| `path` | The PDF in the volume |
| `part_no`, `page_offset` | Which part of the PDF (bronze cuts long PDFs into parts of 50 pages), and how many pages come before it |
| `parsed` | The parser's output (VARIANT): the `elements` in reading order, each with a `type` (title, section_header, text, table as HTML, ...), its `content` and its page in `bbox[0].page_id`. The real page in the PDF is `page_offset + page_id + 1` |
| `parsed_at` | When it was parsed |

**A stream** (`trigger(availableNow=True)`): a run parses the parts it has not seen, `PDFS_PER_BATCH` PDFs at a time
(bronze writes one file per PDF), and stops. The checkpoint remembers what was read, so a rerun only parses new parts.

Written to [doc/code_standard.md](../../doc/code_standard.md).

In [ ]:
from pyspark.sql import functions as F

In [ ]:
CATALOG = "workspace"
BRONZE_SCHEMA = f"{CATALOG}.finance_bench_bronze"
SILVER_SCHEMA_NAME = "finance_bench_silver"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"

PARTS_TABLE = f"{BRONZE_SCHEMA}.bronze_pdf_parts"
PARSED_TABLE = f"{SILVER_SCHEMA}.silver_parsed"
CHECKPOINT = f"/Volumes/{CATALOG}/{SILVER_SCHEMA_NAME}/checkpoints/parse"

PDFS_PER_BATCH = 2  # small, so the parser's rate limit is not hit

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SILVER_SCHEMA}.checkpoints")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {PARSED_TABLE} (
  doc_name STRING, path STRING, part_no INT, page_offset INT, parsed VARIANT, parsed_at TIMESTAMP
)""")

## Step 1: read the parts as a stream and parse them

Each new part goes through `ai_parse_document` once and is appended to `silver_parsed`.

**Example.** `bronze_pdf_parts` has 5 new parts: 4 of `PGE_2020_10K.pdf` and 1 of `AMD_2022_10K.pdf`.

**1a. After `readStream`.** The new rows of `bronze_pdf_parts`, `PDFS_PER_BATCH` PDF files at a time:

| path | part_no | page_offset | pages | content | split_at |
| --- | --- | --- | --- | --- | --- |
| …/pdfs/PGE_2020_10K.pdf | 1 | 0 | 50 | `25 50 44 46 …` | 2026-11-20 09:20:11 |
| …/pdfs/PGE_2020_10K.pdf | 2 | 50 | 50 | `25 50 44 46 …` | 2026-11-20 09:20:11 |
| … | | | | | |
| …/pdfs/AMD_2022_10K.pdf | 1 | 0 | 42 | `25 50 44 46 …` | 2026-11-20 09:20:11 |

**1b. After `select`, appended to `silver_parsed`.** The document name comes from the file name, and the bytes
become the parser's output:

| doc_name | path | part_no | page_offset | parsed | parsed_at |
| --- | --- | --- | --- | --- | --- |
| PGE_2020_10K | …/pdfs/PGE_2020_10K.pdf | 1 | 0 | `{"document": {"elements": [{"id": 0, "type": "title", "content": "UNITED STATES⏎SECURITIES AND EXCHANGE COMMISSION", "bbox": [{"page_id": 0, …}]}, {"id": 1, "type": "text", "content": "Washington, D.C. 20549", …}, …]}, …}` | 2026-11-20 09:31:47 |
| PGE_2020_10K | …/pdfs/PGE_2020_10K.pdf | 2 | 50 | `{"document": {"elements": [{"id": 0, "type": "section_header", "content": "Item 7. …", "bbox": [{"page_id": 0, …}]}, …]}, …}` | 2026-11-20 09:31:47 |
| … | | | | | |
| AMD_2022_10K | …/pdfs/AMD_2022_10K.pdf | 1 | 0 | `{"document": {"elements": […]}, …}` | 2026-11-20 09:32:05 |

In part 2, `page_id` 0 is page 51 of the PDF (`page_offset` 50 + 0 + 1).

In [ ]:
(
    spark.readStream.option("maxFilesPerTrigger", PDFS_PER_BATCH).table(PARTS_TABLE)
    .select(
        F.regexp_extract("path", r"([^/]+)\.pdf$", 1).alias("doc_name"),  # .../pdfs/3M_2018_10K.pdf -> 3M_2018_10K
        "path", "part_no", "page_offset",
        F.expr("ai_parse_document(content, map('version', '2.0'))").alias("parsed"),
        F.current_timestamp().alias("parsed_at"),
    )
    .writeStream.option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(PARSED_TABLE)
    .awaitTermination()
)

In [ ]:
display(spark.sql(f"SELECT count(*) AS parts, count(DISTINCT doc_name) AS documents FROM {PARSED_TABLE}"))